 Numerical transforms from scratch

In [ ]:
import math 

def min_max_solutions(values):
    X_min=min(values)
    X_max=max(values)
    if X_min==X_max:
        return [0.0] * len(values)
    return [(x-X_min)/(X_max-X_min) for x in values]

def standardize(values):
    n=len(values)
    mean=sum(values)/n
    variance=sum((v-mean)**2 for v in values)
    std=math.sqrt(variance) if variance > 0 else 1.0
    return [(x-mean)/std for x in values]

def log_transform(values):
    return [math.log(v+1) for v in values]

def bin_value(values,n_bins=5):
    min_val=min(values)
    max_val=max(values)
    bin_width=(max_val - min_val)/n_bins
    if bin_width==0:
        return [0] * len(values)
    result=[]
    for v in values:
        bin_idx=int((v-min_val)/bin_width)
        bin_idx=min(bin_idx,n_bins-1)
        result.append(bin_idx)
        
    return result

def polynomial_features(row, degree=2):
    n = len(row)
    result = list(row)
    if degree >= 2:
        for i in range(n):
            result.append(row[i] ** 2)
        for i in range(n):
            for j in range(i + 1, n):
                result.append(row[i] * row[j])
    return result

In [ ]:
# 1. min_max_solutions
print(min_max_solutions([10, 20, 30, 40, 50]))
# Expected: [0.0, 0.25, 0.5, 0.75, 1.0]
print(min_max_solutions([5, 5, 5]))
# Expected: [0.0, 0.0, 0.0]  (edge case)

# 2. standardize
print(standardize([1, 2, 3, 4, 5]))
# mean=3, variance=10, std≈3.1623
# Expected: [-1.2649, -0.6325, 0.0, 0.6325, 1.2649]
print(standardize([7, 7, 7]))
# Expected: [0.0, 0.0, 0.0]  (std fallback to 1.0)

# 3. log_transform
print(log_transform([0, 1, 9, 99]))
# log(1)=0, log(2)≈0.693, log(10)≈2.303, log(100)≈4.605
# Expected: [0.0, 0.6931, 2.3026, 4.6052]

# 4. bin_value
print(bin_value([0, 2, 4, 6, 8, 10], n_bins=5))
# Expected: [0, 1, 2, 3, 4, 4]   
print(bin_value([3, 3, 3], n_bins=5))
# Expected: [0, 0, 0]  (edge case)

# 5. polynomial_features
print(polynomial_features([2, 3], degree=2))
# [2, 3, 4, 9, 6]   → original + squares + cross-term (2*3)

[0.0, 0.25, 0.5, 0.75, 1.0]
[0.0, 0.0, 0.0]
[-0.6324555320336759, -0.31622776601683794, 0.0, 0.31622776601683794, 0.6324555320336759]
[0.0, 0.0, 0.0]
[0.0, 0.6931471805599453, 2.302585092994046, 4.605170185988092]
[0, 1, 2, 3, 4, 4]
[0, 0, 0]
[2, 3, 4, 9, 6]


Categorical encoding from scratch

In [1]:
def one_hot_encode(values):
    categories = sorted(set(values))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    n_cats = len(categories)

    encoded = []
    for v in values:
        row = [0] * n_cats
        row[cat_to_idx[v]] = 1
        encoded.append(row)

    return encoded, categories


def label_encode(values):
    categories = sorted(set(values))
    cat_to_int = {cat: i for i, cat in enumerate(categories)}
    return [cat_to_int[v] for v in values], cat_to_int


def target_encode(feature_values, target_values, smoothing=10):
    global_mean = sum(target_values) / len(target_values)

    category_stats = {}
    for feat, target in zip(feature_values, target_values):
        if feat not in category_stats:
            category_stats[feat] = {"sum": 0.0, "count": 0}
        category_stats[feat]["sum"] += target
        category_stats[feat]["count"] += 1

    encoding = {}
    for cat, stats in category_stats.items():
        cat_mean = stats["sum"] / stats["count"]
        weight = stats["count"] / (stats["count"] + smoothing)
        encoding[cat] = weight * cat_mean + (1 - weight) * global_mean

    return [encoding[v] for v in feature_values], encoding

Text features from scratch

In [2]:
def count_vectorize(documents):
    vocab = {}
    idx = 0
    for doc in documents:
        for word in doc.lower().split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    vectors = []
    for doc in documents:
        vec = [0] * len(vocab)
        for word in doc.lower().split():
            vec[vocab[word]] += 1
        vectors.append(vec)

    return vectors, vocab


def tfidf(documents):
    n_docs = len(documents)

    vocab = {}
    idx = 0
    for doc in documents:
        for word in doc.lower().split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    doc_freq = {}
    for doc in documents:
        seen = set()
        for word in doc.lower().split():
            if word not in seen:
                doc_freq[word] = doc_freq.get(word, 0) + 1
                seen.add(word)

    vectors = []
    for doc in documents:
        words = doc.lower().split()
        word_count = len(words)
        tf_map = {}
        for word in words:
            tf_map[word] = tf_map.get(word, 0) + 1

        vec = [0.0] * len(vocab)
        for word, count in tf_map.items():
            tf = count / word_count
            idf = math.log(n_docs / doc_freq[word])
            vec[vocab[word]] = tf * idf
        vectors.append(vec)

    return vectors, vocab